In [15]:
!pip install langchain langchain_community langchain_groq groq

In [16]:
import os
os.environ["groq_AI_API_KEY"]="API_KEY"
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [30]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str)-> float:
  """This functions fetches the conversion factor from the API and returns it"""
  # Simulating API response due to access restriction
  if base_currency == 'USD' and target_currency == 'INR':
    return {'conversion_rate': 83.5} # Hardcoded for demonstration
  else:
    # Original API call, which is currently restricted
    # url=f'http://api.exchangeratesapi.io/v1/convert?access_key=KEY&from={base_currency}&to={target_currency}'
    # response=requests.get(url)
    # return response.json()
    return {'error': 'API access restricted, only USD to INR conversion simulated.'}

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """This tool converts the base currency to the desired value using the conversion rate fetched by another tool"""
  return base_currency_value*conversion_rate

In [31]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency':'INR'})

{'conversion_rate': 83.5}

In [33]:
from groq import Groq
llm = ChatGroq(
    groq_api_key=os.getenv("groq_AI_API_KEY"),
    model_name=os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
    temperature=0,
)
llm_with_tools=llm.bind_tools([get_conversion_factor,convert])


In [34]:
messages=[HumanMessage("What is conversion Rate of USD to INR and can u convert 25 USD to INR")]
llm_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '645j8g1by', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'nxs7k2nen', 'function': {'arguments': '{"base_currency_value":25}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 336, 'total_tokens': 374, 'completion_time': 0.062543705, 'completion_tokens_details': None, 'prompt_time': 0.033001353, 'prompt_tokens_details': None, 'queue_time': 0.062705605, 'total_time': 0.095545058}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4930-7955-7b41-b696-4f18e0a08d45-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '645j8g1by', 'type': 'tool_call'}, {'name'

In [35]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [36]:
import json

for tool_call in ai_message.tool_calls:
  #execute the first tool and get the conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 =get_conversion_factor.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)
  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [37]:
messages

[HumanMessage(content='What is conversion Rate of USD to INR and can u convert 25 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'by6jb5sww', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': '110f2y9s6', 'function': {'arguments': '{"base_currency_value":25}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 336, 'total_tokens': 374, 'completion_time': 0.071553365, 'completion_tokens_details': None, 'prompt_time': 0.032221593, 'prompt_tokens_details': None, 'queue_time': 0.061855835, 'total_time': 0.103774958}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4930-83ac-74c3-92c2-5e3cd39eba9a-0', tool_calls=[{

In [39]:
llm_with_tools.invoke(messages).content

'The conversion rate of USD to INR is 83.5. Converting 25 USD to INR results in 2087.5 INR.'